In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import os
import sys

# ==========================================================
# 1. ROBUST PATH DETECTION (Works in Notebook & Script contexts)
# ==========================================================
# Instead of trusting getcwd(), find the absolute root path based on project folder structure
current_path = os.path.abspath(os.path.dirname(__file__)) if '__file__' in locals() else os.getcwd()

# Walk up until we hit the 'MachLeData' root folder directory
while os.path.basename(current_path) != "MachLeData" and current_path != os.path.dirname(current_path):
    current_path = os.path.dirname(current_path)
    
BASE_DIR = current_path
REPO_PATH = os.path.join(BASE_DIR, 'src', 'models', 'Depth-Anything-V2')

if REPO_PATH not in sys.path:
    sys.path.append(REPO_PATH)

from depth_anything_v2.dpt import DepthAnythingV2

# ==========================================================
# 2. MODEL DEFINITIONS
# ==========================================================
class TrainableDecoder(nn.Module):
    def __init__(self, in_channels=384): 
        super(TrainableDecoder, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, 128, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(128, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 1, kernel_size=3, padding=1)
        
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=True)
        x = F.relu(self.conv2(x))
        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=True)
        x = self.conv3(x) 
        return x

class HybridDepthModel(nn.Module):
    def __init__(self, encoder='vitb'):
        super(HybridDepthModel, self).__init__()
        
        model_configs = {
            'vits': {'encoder': 'vits', 'features': 64, 'out_channels': [48, 96, 192, 384]},
            'vitb': {'encoder': 'vitb', 'features': 128, 'out_channels': [96, 192, 384, 768]}
        }
        self.backbone = DepthAnythingV2(**model_configs[encoder])
        
        for param in self.backbone.parameters():
            param.requires_grad = False
        print("--- Console: DA-V2 Backbone Frozen ---")

        # --- FIX APPLIED HERE ---
        # ViT-B intermediate features are 768 dimensions wide, while ViT-S uses 384.
        in_features = 768 if encoder == 'vitb' else 384
        self.custom_decoder = TrainableDecoder(in_channels=in_features)

    def forward(self, x):
        # Extract features from the intermediate layers of the ViT backbone
        features = self.backbone.pretrained.get_intermediate_layers(x, n=1)[0]
        
        # Reshape the sequential tokens (B, L, D) into a 2D CNN-compatible tensor (B, D, H, W)
        B, L, D = features.shape
        H_feat = int(x.shape[2] / 14) # Assuming patch size 14
        W_feat = int(x.shape[3] / 14)
        features = features.transpose(1, 2).reshape(B, D, H_feat, W_feat)
        
        depth_prediction = self.custom_decoder(features)
        depth_prediction = F.interpolate(depth_prediction, size=x.shape[2:], mode='bilinear', align_corners=True)
        return depth_prediction

def load_hybrid_model(encoder='vitb', device='cuda'):
    model = HybridDepthModel(encoder=encoder)
    ckpt_path = os.path.join(REPO_PATH, 'checkpoints', f'depth_anything_v2_{encoder}.pth')
    
    if os.path.exists(ckpt_path):
        state_dict = torch.load(ckpt_path, map_location='cpu')
        model.backbone.load_state_dict(state_dict)
        print(f"--- Console: Backbone weights loaded from {ckpt_path} ---")
    else:
        print(f"--- Warning: Checkpoint not found at {ckpt_path} ---")
        
    return model.to(device)

if __name__ == "__main__":
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = load_hybrid_model(device=device)
    print("Hybrid Model Initialized Successfully.")

--- Console: DA-V2 Backbone Frozen ---
--- Console: Backbone weights loaded from C:\Users\kanha\Documents\MachLeData\src\models\Depth-Anything-V2\checkpoints\depth_anything_v2_vitb.pth ---
Hybrid Model Initialized Successfully.
